# Comparação entre splits e resoluções

## Objetivo

Compara Dice, tempo e sucesso dos óstios entre splits e resoluções.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent,
    NOTEBOOK_CWD.parent.parent,
):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()


In [ ]:
from utils.project.notebook_env import get_default_split_paths
import matplotlib.pyplot as plt
import seaborn as sns

import utils.comparison_utils as cmp
import utils.visualization as viz

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="deep",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 110,
        "savefig.dpi": 300,
    },
)
plt.close("all")

SPLIT_PATHS_BY_RESOLUTION = get_default_split_paths(REPO_ROOT)

VALID_SPLITS = ("train", "val", "test")


## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento de dados

### Dados de train, test e validação

In [ ]:
subset_summary_df = cmp.build_split_resolution_summary(
    SPLIT_PATHS_BY_RESOLUTION,
    valid_splits=VALID_SPLITS,
)
display(subset_summary_df)

missing_rows = subset_summary_df[~subset_summary_df["is_available"]]
if not missing_rows.empty:
    print("Combinations without data:")
    display(missing_rows[["subset", "resolution"]])

## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Resultados por subset: val, train e test

In [ ]:
ax_subset = viz.plot_subset_metric_by_resolution(
    subset_summary_df=subset_summary_df,
    metric_col="dice_medio_correto",
    title="Dice médio (both_correct + both_tolerable) por conjunto e resolução",
    ylabel="Dice médio",
    palette=["#4C78A8", "#F58518"],
    ylim=(0, 1),
    bar_label_fmt="%.3f",
)

In [ ]:
ax_subset_all = viz.plot_subset_metric_by_resolution(
    subset_summary_df=subset_summary_df,
    metric_col="dice_medio_todos",
    title="Dice médio (todos os resultados, incluindo casos ruins) por conjunto e resolução",
    ylabel="Dice médio",
    palette=["#4C78A8", "#F58518"],
    ylim=(0, 1),
    bar_label_fmt="%.3f",
)

### Impacto dos casos ruins no Dice

In [ ]:
subset_summary_df["delta_dice"] = (
    subset_summary_df["dice_medio_todos"] - subset_summary_df["dice_medio_correto"]
)

ax_subset_delta = viz.plot_subset_metric_by_resolution(
    subset_summary_df=subset_summary_df,
    metric_col="delta_dice",
    title="Impacto dos casos ruins no Dice por conjunto e resolução",
    ylabel="Dice médio (todos - corretos)",
    palette=["#9C755F", "#B279A2"],
    hline_y=0,
    bar_label_fmt="%.3f",
)

### Tempo de execução por conjunto

In [ ]:
ax_subset_time = viz.plot_subset_execution_time_by_resolution(
    subset_summary_df=subset_summary_df,
    palette=["#F58518", "#E45756"],
)

### Percentual de óstios detectados com sucesso por conjunto

In [ ]:
ax_subset_success = viz.plot_subset_ostia_success_by_resolution(
    subset_summary_df=subset_summary_df,
    palette=["#54A24B", "#2E8B57"],
)

## Conclusão

A tabela consolidada evidencia as combinações disponíveis e permite comparar Dice, tempo e sucesso dos óstios entre splits e resoluções.